# Module 4 project-exercise demo — Configuration-derived frequencies and multi-axis rotation

**Implementation demo, not a completed full-scale project.** Exact PI/static NTK/YaRN and sectioned M-RoPE on small inputs. These are geometry checks, with no training or model-quality measurement. The gpt-oss configuration is pinned by revision and checksum.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'


## Demo 1. Recover the spectrum from a configuration

A positional spectrum can be reconstructed without loading model weights. Start by verifying the saved `config.json` bytes, then read the head width, base, reference length, and extension factor. The functions below build the original spectrum, PI, a static NTK-aware transformation, and the YaRN ramp. They come from the main notebook; the ALiBi and M-RoPE helpers support the next experiment.

On the plot, locate the unchanged first NTK pair and the YaRN transition. Also distinguish the amplitude applied to each rotary vector from its combined effect on a query–key dot product.

In [ ]:
import importlib.util
import subprocess
import sys
for package in ('numpy', 'matplotlib'):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import hashlib
import json
from dataclasses import dataclass
from numbers import Integral, Real
from typing import Sequence


def _positive_int(name: str, value: object) -> int:
    """Return a strictly positive integer and reject booleans explicitly."""

    if isinstance(value, bool) or not isinstance(value, Integral):
        raise TypeError(f"{name} must be an integer")
    result = int(value)
    if result <= 0:
        raise ValueError(f"{name} must be positive")
    return result


def _positive_even_int(name: str, value: object, *, minimum: int = 2) -> int:
    """Return an even integer not smaller than ``minimum``."""

    result = _positive_int(name, value)
    if result < minimum:
        raise ValueError(f"{name} must be at least {minimum}")
    if result % 2:
        raise ValueError(f"{name} must be even")
    return result


def _finite_real(name: str, value: object) -> float:
    """Return a finite real scalar and reject booleans and complex values."""

    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError(f"{name} must be a real scalar")
    result = float(value)
    if not math.isfinite(result):
        raise ValueError(f"{name} must be finite")
    return result


def _real_greater_than(name: str, value: object, lower: float) -> float:
    """Return a finite scalar strictly greater than ``lower``."""

    result = _finite_real(name, value)
    if result <= lower:
        raise ValueError(f"{name} must be greater than {lower}")
    return result


def _finite_array(
    name: str,
    value: object,
    *,
    ndim: int | None = None,
    minimum_ndim: int | None = None,
) -> np.ndarray:
    """Convert to float64 and enforce rank, non-emptiness, and finite values."""

    try:
        if np.iscomplexobj(value):
            raise TypeError(f"{name} must be real")
        array = np.asarray(value, dtype=np.float64)
    except (TypeError, ValueError) as exc:
        raise TypeError(f"{name} must be numeric") from exc
    if ndim is not None and array.ndim != ndim:
        raise ValueError(f"{name} must have rank {ndim}, got shape {array.shape}")
    if minimum_ndim is not None and array.ndim < minimum_ndim:
        raise ValueError(f"{name} must have rank at least {minimum_ndim}, got shape {array.shape}")
    if array.size == 0:
        raise ValueError(f"{name} must be non-empty")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{name} must contain only finite values")
    return array


def _position_vector(name: str, value: object) -> np.ndarray:
    """Validate a non-empty one-dimensional vector of non-negative integer positions."""

    raw = np.asarray(value)
    if raw.ndim != 1 or raw.size == 0:
        raise ValueError(f"{name} must be a non-empty one-dimensional vector")
    if raw.dtype.kind not in "iu":
        raise TypeError(f"{name} must contain integer positions")
    result = raw.astype(np.int64, copy=False)
    if np.any(result < 0):
        raise ValueError(f"{name} must contain non-negative positions")
    return result


def _inverse_frequency_vector(value: object, *, expected_pairs: int | None = None) -> np.ndarray:
    """Validate a positive one-dimensional inverse-frequency vector."""

    inv = _finite_array("inverse_frequencies", value, ndim=1)
    if np.any(inv <= 0.0):
        raise ValueError("inverse_frequencies must be strictly positive")
    if expected_pairs is not None and inv.size != expected_pairs:
        raise ValueError(
            f"inverse_frequencies must have {expected_pairs} entries, got {inv.size}"
        )
    return inv


def _sequence_tensor(name: str, value: object) -> np.ndarray:
    """Validate a finite sequence tensor with shape ``[..., sequence, head_dim]``."""

    tensor = _finite_array(name, value, minimum_ndim=2)
    _positive_even_int(f"{name}.shape[-1]", tensor.shape[-1])
    if tensor.shape[-2] <= 0:
        raise ValueError(f"{name} must have a non-empty sequence axis")
    return tensor

def rope_inverse_frequencies(head_dim: int, base: Real = 10_000.0) -> np.ndarray:
    """Return ``base**(-2i/head_dim)`` for the ``head_dim/2`` RoPE pairs."""

    dim = _positive_even_int("head_dim", head_dim)
    rope_base = _real_greater_than("base", base, 1.0)
    pair_indices = np.arange(dim // 2, dtype=np.float64)
    inv = rope_base ** (-2.0 * pair_indices / dim)
    if not np.all(np.isfinite(inv)) or np.any(inv <= 0.0):
        raise FloatingPointError("inverse-frequency construction produced invalid values")
    return inv


def sinusoidal_encoding(
    positions: object,
    model_dim: int,
    *,
    base: Real = 10_000.0,
) -> np.ndarray:
    """Return the classic sinusoidal encoding with shape ``[positions, model_dim]``."""

    pos = _position_vector("positions", positions)
    dim = _positive_even_int("model_dim", model_dim)
    inv = rope_inverse_frequencies(dim, base)
    angles = pos.astype(np.float64)[:, None] * inv[None, :]
    result = np.empty((pos.size, dim), dtype=np.float64)
    result[:, 0::2] = np.sin(angles)
    result[:, 1::2] = np.cos(angles)
    if not np.all(np.isfinite(result)):
        raise FloatingPointError("sinusoidal encoding produced non-finite values")
    return result


def wavelengths_from_inverse_frequencies(inverse_frequencies: object) -> np.ndarray:
    """Convert radians-per-position frequencies to tokens per full ``2π`` turn."""

    inv = _inverse_frequency_vector(inverse_frequencies)
    return (2.0 * math.pi) / inv


def rotations_within_context(
    inverse_frequencies: object,
    context_length: int,
) -> np.ndarray:
    """Return turns (including fractions) across a span L = context_length.

    This is the YaRN L*theta/(2*pi) diagnostic. Positions 0..L-1 have a maximum
    endpoint separation of L-1, which is a different convention.
    """

    inv = _inverse_frequency_vector(inverse_frequencies)
    length = _positive_int("context_length", context_length)
    return length * inv / (2.0 * math.pi)


def full_turn_count(inverse_frequencies: object, context_length: int) -> int:
    """Count frequency pairs that complete at least one full turn in the context."""

    return int(np.count_nonzero(rotations_within_context(inverse_frequencies, context_length) >= 1.0))

def rotate_half_split(x: object) -> np.ndarray:
    """Apply the split-half quarter turn used by HF/Llama-style RoPE implementations."""

    values = _sequence_tensor("x", x)
    half = values.shape[-1] // 2
    return np.concatenate((-values[..., half:], values[..., :half]), axis=-1)


def _angles_for_sequence(
    x: np.ndarray,
    positions: np.ndarray,
    inverse_frequencies: np.ndarray,
) -> np.ndarray:
    """Build a broadcastable angle tensor for ``[..., T, D]`` sequence tensors."""

    if positions.size != x.shape[-2]:
        raise ValueError(
            f"positions length {positions.size} does not match sequence axis {x.shape[-2]}"
        )
    if inverse_frequencies.size != x.shape[-1] // 2:
        raise ValueError("inverse_frequencies do not match head_dim")
    base_angles = positions.astype(np.float64)[:, None] * inverse_frequencies[None, :]
    return base_angles.reshape((1,) * (x.ndim - 2) + base_angles.shape)


def apply_rope_adjacent(
    x: object,
    positions: object,
    inverse_frequencies: object,
) -> np.ndarray:
    """Apply RoPE to adjacent pairs ``(x_0,x_1), (x_2,x_3), ...``."""

    values = _sequence_tensor("x", x)
    pos = _position_vector("positions", positions)
    inv = _inverse_frequency_vector(
        inverse_frequencies,
        expected_pairs=values.shape[-1] // 2,
    )
    angles = _angles_for_sequence(values, pos, inv)
    cosine = np.cos(angles)
    sine = np.sin(angles)
    first = values[..., 0::2]
    second = values[..., 1::2]
    result = np.empty_like(values, dtype=np.float64)
    result[..., 0::2] = cosine * first - sine * second
    result[..., 1::2] = sine * first + cosine * second
    if not np.all(np.isfinite(result)):
        raise FloatingPointError("adjacent-pair RoPE produced non-finite values")
    return result


def apply_rope_split_half(
    x: object,
    positions: object,
    inverse_frequencies: object,
) -> np.ndarray:
    """Apply split-half RoPE to pairs ``(x_j, x_{j+D/2})``."""

    values = _sequence_tensor("x", x)
    pos = _position_vector("positions", positions)
    inv = _inverse_frequency_vector(
        inverse_frequencies,
        expected_pairs=values.shape[-1] // 2,
    )
    angles = _angles_for_sequence(values, pos, inv)
    repeated = np.concatenate((angles, angles), axis=-1)
    result = values * np.cos(repeated) + rotate_half_split(values) * np.sin(repeated)
    if not np.all(np.isfinite(result)):
        raise FloatingPointError("split-half RoPE produced non-finite values")
    return result


def apply_rope_complex_adjacent(
    x: object,
    positions: object,
    inverse_frequencies: object,
) -> np.ndarray:
    """Apply adjacent-pair RoPE through the equivalent complex multiplication."""

    values = _sequence_tensor("x", x)
    pos = _position_vector("positions", positions)
    inv = _inverse_frequency_vector(
        inverse_frequencies,
        expected_pairs=values.shape[-1] // 2,
    )
    angles = _angles_for_sequence(values, pos, inv)
    complex_values = values[..., 0::2] + 1j * values[..., 1::2]
    rotated = complex_values * np.exp(1j * angles)
    result = np.empty_like(values, dtype=np.float64)
    result[..., 0::2] = rotated.real
    result[..., 1::2] = rotated.imag
    if not np.all(np.isfinite(result)):
        raise FloatingPointError("complex RoPE produced non-finite values")
    return result


def adjacent_to_split_permutation(head_dim: int) -> np.ndarray:
    """Return the coordinate permutation mapping adjacent pairs to split-half pairs."""

    dim = _positive_even_int("head_dim", head_dim)
    return np.concatenate((np.arange(0, dim, 2), np.arange(1, dim, 2)))


def rope_attention_logits(
    queries: object,
    keys: object,
    query_positions: object,
    key_positions: object,
    inverse_frequencies: object,
    *,
    pairing: str = "split_half",
) -> np.ndarray:
    """Return unscaled rotated dot products ``[..., query, key]``.

    Divide by sqrt(head_dim) before the standard attention softmax. The historical
    function name and fixture key 'logits' are retained for file compatibility.
    """

    q = _sequence_tensor("queries", queries)
    k = _sequence_tensor("keys", keys)
    if q.ndim != k.ndim or q.shape[:-2] != k.shape[:-2]:
        raise ValueError("queries and keys must have identical batch dimensions")
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("queries and keys must have the same head_dim")
    if pairing == "split_half":
        apply = apply_rope_split_half
    elif pairing == "adjacent":
        apply = apply_rope_adjacent
    else:
        raise ValueError("pairing must be 'split_half' or 'adjacent'")
    q_rot = apply(q, query_positions, inverse_frequencies)
    k_rot = apply(k, key_positions, inverse_frequencies)
    return np.matmul(q_rot, np.swapaxes(k_rot, -1, -2))

def position_interpolated_frequencies(
    inverse_frequencies: object,
    factor: Real,
) -> np.ndarray:
    """Apply position interpolation by dividing every frequency by ``factor``."""

    inv = _inverse_frequency_vector(inverse_frequencies)
    scale = _finite_real("factor", factor)
    if scale < 1.0:
        raise ValueError("factor must be at least 1")
    return inv / scale


def ntk_scaled_base(head_dim: int, base: Real, factor: Real) -> float:
    """Return the static NTK-aware base ``base * factor**(d/(d-2))``."""

    dim = _positive_even_int("head_dim", head_dim, minimum=4)
    rope_base = _real_greater_than("base", base, 1.0)
    scale = _finite_real("factor", factor)
    if scale < 1.0:
        raise ValueError("factor must be at least 1")
    result = rope_base * scale ** (dim / (dim - 2))
    if not math.isfinite(result):
        raise FloatingPointError("NTK base is non-finite")
    return result


def ntk_scaled_frequencies(head_dim: int, base: Real, factor: Real) -> np.ndarray:
    """Return inverse frequencies generated from the static NTK-aware base."""

    return rope_inverse_frequencies(head_dim, ntk_scaled_base(head_dim, base, factor))


@dataclass(frozen=True)
class YarnSpectrum:
    """Piecewise YaRN spectrum plus diagnostic boundaries and scaling metadata."""

    inverse_frequencies: np.ndarray
    original_inverse_frequencies: np.ndarray
    ramp: np.ndarray
    low: float
    high: float
    truncate: bool
    kept_pairs: int
    transition_pairs: int
    interpolated_pairs: int
    attention_factor: float
    score_factor_if_applied_to_q_and_k: float


def yarn_scaled_frequencies(
    head_dim: int,
    base: Real,
    factor: Real,
    original_context: int,
    *,
    beta_fast: Real = 32.0,
    beta_slow: Real = 1.0,
    truncate: bool = False,
) -> YarnSpectrum:
    """Construct the lecture's piecewise YaRN frequency spectrum.

    ``ramp == 0`` keeps a frequency, ``ramp == 1`` divides it by ``factor``.
    With ``truncate=False`` this follows the continuous correction bounds used by the
    original gpt-oss implementation. ``truncate=True`` reproduces the integer-boundary
    option supported by the generic Hugging Face YaRN utility.
    """

    dim = _positive_even_int("head_dim", head_dim, minimum=4)
    rope_base = _real_greater_than("base", base, 1.0)
    scale = _finite_real("factor", factor)
    if scale < 1.0:
        raise ValueError("factor must be at least 1")
    context = _positive_int("original_context", original_context)
    fast = _real_greater_than("beta_fast", beta_fast, 0.0)
    slow = _real_greater_than("beta_slow", beta_slow, 0.0)
    if fast <= slow:
        raise ValueError("beta_fast must be greater than beta_slow")
    if not isinstance(truncate, bool):
        raise TypeError("truncate must be a bool")

    original = rope_inverse_frequencies(dim, rope_base)
    pair_count = dim // 2

    def correction_dimension(rotations: float) -> float:
        """Map a target turn count to the corresponding continuous pair index."""

        return dim * math.log(context / (rotations * 2.0 * math.pi)) / (2.0 * math.log(rope_base))

    low_raw = correction_dimension(fast)
    high_raw = correction_dimension(slow)
    low = math.floor(low_raw) if truncate else low_raw
    high = math.ceil(high_raw) if truncate else high_raw
    low = max(0.0, min(float(pair_count - 1), float(low)))
    high = max(0.0, min(float(pair_count - 1), float(high)))
    if high <= low:
        raise ValueError("YaRN correction range is empty for this configuration")
    pair_indices = np.arange(pair_count, dtype=np.float64)
    ramp = np.clip((pair_indices - low) / (high - low), 0.0, 1.0)
    scaled = original * (1.0 - ramp) + (original / scale) * ramp
    attention_factor = 0.1 * math.log(scale) + 1.0 if scale > 1.0 else 1.0
    score_factor_if_applied_to_q_and_k = attention_factor**2
    kept = int(np.count_nonzero(ramp == 0.0))
    interpolated = int(np.count_nonzero(ramp == 1.0))
    transition = pair_count - kept - interpolated
    return YarnSpectrum(
        inverse_frequencies=scaled,
        original_inverse_frequencies=original,
        ramp=ramp,
        low=low,
        high=high,
        truncate=truncate,
        kept_pairs=kept,
        transition_pairs=transition,
        interpolated_pairs=interpolated,
        attention_factor=attention_factor,
        score_factor_if_applied_to_q_and_k=score_factor_if_applied_to_q_and_k,
    )

def alibi_bias(
    slopes: object,
    query_positions: object,
    key_positions: object,
    *,
    causal: bool = True,
) -> np.ndarray:
    """Return an ALiBi tensor with shape ``[heads, query, key]``.

    In causal mode, future keys receive ``-inf`` and allowed keys receive
    ``-slope * (query_position - key_position)``. In bidirectional mode the absolute
    distance is used.
    """

    if not isinstance(causal, bool):
        raise TypeError("causal must be a bool")
    slope_values = _finite_array("slopes", slopes, ndim=1)
    if np.any(slope_values <= 0.0):
        raise ValueError("slopes must be strictly positive")
    q_pos = _position_vector("query_positions", query_positions)
    k_pos = _position_vector("key_positions", key_positions)
    delta = q_pos[:, None] - k_pos[None, :]
    if causal:
        allowed = delta >= 0
        base = -np.abs(delta).astype(np.float64)
        bias = slope_values[:, None, None] * base[None, :, :]
        bias = np.where(allowed[None, :, :], bias, -np.inf)
    else:
        bias = -slope_values[:, None, None] * np.abs(delta)[None, :, :]
    return bias


def apply_mrope_adjacent(
    x: object,
    coordinates: object,
    axis_sections: Sequence[int],
    inverse_frequencies: object,
) -> np.ndarray:
    """Apply adjacent-pair multi-axis RoPE.

    ``coordinates`` has shape ``[sequence, axes]``. ``axis_sections`` gives the number of
    frequency pairs assigned to each axis and must sum to ``head_dim / 2``.
    """

    values = _sequence_tensor("x", x)
    raw_coordinates = np.asarray(coordinates)
    if raw_coordinates.ndim != 2 or raw_coordinates.shape[0] != values.shape[-2]:
        raise ValueError("coordinates must have shape [sequence, axes]")
    if raw_coordinates.dtype.kind not in "iu":
        raise TypeError("coordinates must contain integer positions")
    coords = raw_coordinates.astype(np.int64, copy=False)
    if np.any(coords < 0):
        raise ValueError("coordinates must be non-negative")

    if isinstance(axis_sections, (str, bytes)):
        raise TypeError("axis_sections must be a sequence of integers")
    sections: list[int] = []
    for index, value in enumerate(axis_sections):
        sections.append(_positive_int(f"axis_sections[{index}]", value))
    if len(sections) != coords.shape[1]:
        raise ValueError("axis_sections length must match the coordinate axis count")
    pair_count = values.shape[-1] // 2
    if sum(sections) != pair_count:
        raise ValueError("axis_sections must sum to head_dim / 2")

    inv = _inverse_frequency_vector(inverse_frequencies, expected_pairs=pair_count)
    axis_index = np.repeat(np.arange(len(sections)), sections)
    base_angles = coords[:, axis_index].astype(np.float64) * inv[None, :]
    angles = base_angles.reshape((1,) * (values.ndim - 2) + base_angles.shape)
    cosine = np.cos(angles)
    sine = np.sin(angles)
    first = values[..., 0::2]
    second = values[..., 1::2]
    result = np.empty_like(values, dtype=np.float64)
    result[..., 0::2] = cosine * first - sine * second
    result[..., 1::2] = sine * first + cosine * second
    if not np.all(np.isfinite(result)):
        raise FloatingPointError("M-RoPE produced non-finite values")
    return result
INK = '#2E3440'
MUTED = '#4C566A'
BORDER = '#D8DEE9'
BLUE = '#5E81AC'
ORANGE = '#D08770'
PURPLE = '#B48EAD'
GREEN = '#A3BE8C'
PALE = {'blue': '#E8EEF5', 'green': '#EFF4EA', 'purple': '#F3EEF3', 'orange': '#FAF0E8', 'yellow': '#FAF5E3', 'gray': '#ECEFF4'}

def plot_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 12,
        "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
        "axes.titlesize": 17, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": BORDER, "axes.linewidth": .7,
        "axes.facecolor": "white", "figure.facecolor": "white", "savefig.facecolor": "white",
        "grid.color": BORDER, "grid.linewidth": .7, "axes.axisbelow": True,
        "legend.fontsize": 11, "legend.frameon": False, "savefig.dpi": 150,
        "xtick.major.size": 0, "ytick.major.size": 0,
    })

def scaling_plot(reference, lang='en'):
    plot_style()
    ru = lang == 'ru'; sc = reference['scaling']; yr = sc['yarn']; x = np.arange(len(sc['raw']))
    fig, (ax, ramp_ax) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                    height_ratios=[3, 1], layout='constrained')
    series = [('Исходные' if ru else 'Original', sc['raw'], MUTED, '-'),
              ('PI', sc['position_interpolation'], PURPLE, '--'),
              ('Статическая NTK-aware' if ru else 'Static NTK-aware', sc['ntk'], ORANGE, '-.'),
              ('YaRN', yr['inverse_frequencies'], BLUE, '-')]
    for name, values, color, style in series:
        ax.plot(x, values, linestyle=style, color=color, linewidth=2.2, label=name)
    ax.set_yscale('log'); ax.grid(axis='y')
    ax.set_ylabel('Частота, рад/шаг (лог. шкала)' if ru else 'Frequency, rad/step (log scale)')
    ax.set_title('Как меняется спектр · s=32' if ru else 'How the spectrum changes · s=32', pad=16)
    ax.legend(loc='lower left', ncol=2)
    ramp_ax.plot(x, yr['ramp'], color=BLUE, marker='o', markersize=3)
    for value in (yr['low'], yr['high']):
        ramp_ax.axvline(value, color=MUTED, linestyle=':', linewidth=1.4)
    ramp_ax.set(xlabel='Индекс пары i' if ru else 'Pair index i',
                ylabel='Вес PI' if ru else 'PI weight', ylim=(-.08,1.08), yticks=[0,.5,1])
    ramp_ax.grid(axis='y')
    return fig

reference = json.loads('{\n  "alibi": {\n    "allowed_bias_with_null_for_future": [\n      [\n        [\n          -0.0,\n          null,\n          null,\n          null\n        ],\n        [\n          -1.0,\n          -0.0,\n          null,\n          null\n        ],\n        [\n          -2.0,\n          -1.0,\n          -0.0,\n          null\n        ],\n        [\n          -3.0,\n          -2.0,\n          -1.0,\n          -0.0\n        ]\n      ],\n      [\n        [\n          -0.0,\n          null,\n          null,\n          null\n        ],\n        [\n          -0.5,\n          -0.0,\n          null,\n          null\n        ],\n        [\n          -1.0,\n          -0.5,\n          -0.0,\n          null\n        ],\n        [\n          -1.5,\n          -1.0,\n          -0.5,\n          -0.0\n        ]\n      ]\n    ],\n    "positions": [\n      0,\n      1,\n      2,\n      3\n    ],\n    "slopes": [\n      1.0,\n      0.5\n    ]\n  },\n  "meta": {\n    "description": "Frozen numerical contract independent of lecture prose and figures",\n    "module": "module-04",\n    "version": 2\n  },\n  "mrope": {\n    "axis_sections": [\n      2,\n      1,\n      1\n    ],\n    "base": 10000.0,\n    "head_dim": 8,\n    "image_coordinates": [\n      [\n        0,\n        0,\n        0\n      ],\n      [\n        0,\n        0,\n        1\n      ],\n      [\n        0,\n        1,\n        0\n      ],\n      [\n        0,\n        1,\n        1\n      ]\n    ],\n    "image_input": [\n      [\n        0.0,\n        0.1,\n        0.2,\n        0.3,\n        0.4,\n        0.5,\n        0.6,\n        0.7\n      ],\n      [\n        0.8,\n        0.9,\n        1.0,\n        1.1,\n        1.2,\n        1.3,\n        1.4,\n        1.5\n      ],\n      [\n        1.6,\n        1.7,\n        1.8,\n        1.9,\n        2.0,\n        2.1,\n        2.2,\n        2.3\n      ],\n      [\n        2.4,\n        2.5,\n        2.6,\n        2.7,\n        2.8,\n        2.9,\n        3.0,\n        3.1\n      ]\n    ],\n    "image_output": [\n      [\n        0.0,\n        0.1,\n        0.2,\n        0.3,\n        0.4,\n        0.5,\n        0.6,\n        0.7\n      ],\n      [\n        0.8,\n        0.9,\n        1.0,\n        1.1,\n        1.2,\n        1.3,\n        1.3984993002500583,\n        1.5013992497667292\n      ],\n      [\n        1.6,\n        1.7,\n        1.8,\n        1.9,\n        1.9789003508315806,\n        2.1198946675433303,\n        2.2,\n        2.3\n      ],\n      [\n        2.4,\n        2.5,\n        2.6,\n        2.7,\n        2.7708604844975793,\n        2.9278545345439957,\n        2.996898500516792,\n        3.1029984495001295\n      ]\n    ],\n    "text_coordinates": [\n      [\n        4,\n        4,\n        4\n      ],\n      [\n        5,\n        5,\n        5\n      ],\n      [\n        6,\n        6,\n        6\n      ]\n    ],\n    "text_input": [\n      [\n        0.0,\n        0.1,\n        0.2,\n        0.3,\n        0.4,\n        0.5,\n        0.6,\n        0.7\n      ],\n      [\n        0.8,\n        0.9,\n        1.0,\n        1.1,\n        1.2,\n        1.3,\n        1.4,\n        1.5\n      ],\n      [\n        1.6,\n        1.7,\n        1.8,\n        1.9,\n        2.0,\n        2.1,\n        2.2,\n        2.3\n      ]\n    ],\n    "text_matches_standard_1d": true,\n    "text_output": [\n      [\n        0.07568024953079283,\n        -0.06536436208636119,\n        0.0673866961079819,\n        0.3542019666625956,\n        0.37968537557107407,\n        0.5155957870051426,\n        0.5971952074730607,\n        0.7023943936074717\n      ],\n      [\n        1.0899615955674056,\n        -0.5118434528136071,\n        0.3502144694257494,\n        1.444766356683613,\n        1.1335273924220777,\n        1.3583503416382703,\n        1.3924825312864193,\n        1.5069812208724322\n      ],\n      [\n        2.0112788055787596,\n        1.185224690187341,\n        0.4127834073868535,\n        2.5844941204394525,\n        1.8704766662635748,\n        2.216149146822818,\n        2.1861604829186514,\n        2.3131585209243424\n      ]\n    ],\n    "text_positions": [\n      4,\n      5,\n      6\n    ],\n    "text_standard_rope_output": [\n      [\n        0.07568024953079283,\n        -0.06536436208636119,\n        0.0673866961079819,\n        0.3542019666625956,\n        0.37968537557107407,\n        0.5155957870051426,\n        0.5971952074730607,\n        0.7023943936074717\n      ],\n      [\n        1.0899615955674056,\n        -0.5118434528136071,\n        0.3502144694257494,\n        1.444766356683613,\n        1.1335273924220777,\n        1.3583503416382703,\n        1.3924825312864193,\n        1.5069812208724322\n      ],\n      [\n        2.0112788055787596,\n        1.185224690187341,\n        0.4127834073868535,\n        2.5844941204394525,\n        1.8704766662635748,\n        2.216149146822818,\n        2.1861604829186514,\n        2.3131585209243424\n      ]\n    ]\n  },\n  "scaling": {\n    "base": 150000.0,\n    "beta_fast": 32.0,\n    "beta_slow": 1.0,\n    "extended_context": 131072,\n    "factor": 32.0,\n    "head_dim": 64,\n    "ntk": [\n      1.0,\n      0.6161605983892874,\n      0.37965388300744474,\n      0.23392776373468369,\n      0.14413707088263056,\n      0.08881158384512078,\n      0.05472219864590999,\n      0.033717662662841354,\n      0.020775495202624464,\n      0.01280104155588286,\n      0.007887497425078919,\n      0.00485996513323059,\n      0.0029945190246424332,\n      0.001845104634111787,\n      0.0011368807754451658,\n      0.0007005011388955706,\n      0.00043162120091427207,\n      0.00026594797743284077,\n      0.00016386666491543985,\n      0.00010096818231035427,\n      6.221261563062656e-05,\n      3.83329624743296e-05,\n      2.3619261096217023e-05,\n      1.4553258050557898e-05,\n      8.967144188945469e-06,\n      5.525200929303662e-06,\n      3.4044111108207913e-06,\n      2.0976639872064776e-06,\n      1.2924978975768017e-06,\n      7.963862779878181e-07,\n      4.907018455939914e-07,\n      3.0235114281192144e-07\n    ],\n    "ntk_base": 5367776.967415848,\n    "original_context": 4096,\n    "position_interpolation": [\n      0.03125,\n      0.0215326345590051,\n      0.014836939233653148,\n      0.01022330849575739,\n      0.007044312506339714,\n      0.0048538434213901555,\n      0.0033445131711816074,\n      0.0023045177565706506,\n      0.001587914837983171,\n      0.0010941436773476293,\n      0.0007539134706999197,\n      0.0005194797841181623,\n      0.00035794458727063204,\n      0.00024663967968231775,\n      0.00016994566689118156,\n      0.0001171000940817352,\n      8.068715304598785e-05,\n      5.55970233646647e-05,\n      3.8308812373753384e-05,\n      2.639646903147278e-05,\n      1.818833668168956e-05,\n      1.2532569824095001e-05,\n      8.63549587543848e-06,\n      5.950239261491605e-06,\n      4.09997848180298e-06,\n      2.8250668271503398e-06,\n      1.9465962110014815e-06,\n      1.3412910350540445e-06,\n      9.242089502428302e-07,\n      6.36820914615699e-07,\n      4.387978250864396e-07,\n      3.0235114281192144e-07\n    ],\n    "raw": [\n      1.0,\n      0.6890443058881632,\n      0.4747820554769007,\n      0.3271458718642365,\n      0.22541800020287084,\n      0.15532298948448497,\n      0.10702442147781144,\n      0.07374456821026082,\n      0.050813274815461475,\n      0.03501259767512414,\n      0.02412523106239743,\n      0.016623353091781194,\n      0.011454226792660225,\n      0.007892469749834168,\n      0.00543826134051781,\n      0.0037472030106155264,\n      0.0025819888974716113,\n      0.0017791047476692703,\n      0.0012258819959601083,\n      0.0008446870090071289,\n      0.0005820267738140659,\n      0.00040104223437104005,\n      0.00027633586801403137,\n      0.00019040765636773135,\n      0.00013119931141769537,\n      9.040213846881087e-05,\n      6.229107875204741e-05,\n      4.292131312172942e-05,\n      2.9574686407770566e-05,\n      2.0378269267702367e-05,\n      1.4041530402766067e-05,\n      9.675236569981486e-06\n    ],\n    "yarn": {\n      "attention_factor": 1.3465735902799727,\n      "hf_truncated_high": 18,\n      "hf_truncated_low": 8,\n      "high": 17.39802450158856,\n      "interpolated_pairs": 14,\n      "inverse_frequencies": [\n        1.0,\n        0.6890443058881632,\n        0.4747820554769007,\n        0.3271458718642365,\n        0.22541800020287084,\n        0.15532298948448497,\n        0.10702442147781144,\n        0.07374456821026082,\n        0.050813274815461475,\n        0.03170569618466377,\n        0.01933500112654036,\n        0.011592049256286924,\n        0.006794959489732219,\n        0.0038603593171920685,\n        0.0020937923789696887,\n        0.0010526021013863357,\n        0.00045648391922324086,\n        0.0001293187012450632,\n        3.8308812373753384e-05,\n        2.639646903147278e-05,\n        1.818833668168956e-05,\n        1.2532569824095001e-05,\n        8.63549587543848e-06,\n        5.950239261491605e-06,\n        4.09997848180298e-06,\n        2.8250668271503398e-06,\n        1.9465962110014815e-06,\n        1.3412910350540445e-06,\n        9.242089502428302e-07,\n        6.36820914615699e-07,\n        4.387978250864396e-07,\n        3.0235114281192144e-07\n      ],\n      "kept_pairs": 9,\n      "low": 8.092779115512402,\n      "ramp": [\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.0,\n        0.09749564324709936,\n        0.20496191184183662,\n        0.3124281804365739,\n        0.4198944490313112,\n        0.5273607176260484,\n        0.6348269862207857,\n        0.742293254815523,\n        0.8497595234102603,\n        0.9572257920049976,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0,\n        1.0\n      ],\n      "score_factor_if_applied_to_q_and_k": 1.8132604340394958,\n      "transition_pairs": 9,\n      "truncate": false\n    }\n  },\n  "sinusoidal": {\n    "base": 10000.0,\n    "encoding": [\n      [\n        0.0,\n        1.0,\n        0.0,\n        1.0,\n        0.0,\n        1.0,\n        0.0,\n        1.0\n      ],\n      [\n        0.8414709848078965,\n        0.5403023058681398,\n        0.09983341664682815,\n        0.9950041652780258,\n        0.009999833334166664,\n        0.9999500004166653,\n        0.0009999998333333417,\n        0.9999995000000417\n      ],\n      [\n        0.9092974268256817,\n        -0.4161468365471424,\n        0.19866933079506122,\n        0.9800665778412416,\n        0.01999866669333308,\n        0.9998000066665778,\n        0.0019999986666669333,\n        0.9999980000006666\n      ],\n      [\n        0.1411200080598672,\n        -0.9899924966004454,\n        0.2955202066613396,\n        0.955336489125606,\n        0.02999550020249566,\n        0.9995500337489875,\n        0.002999995500002025,\n        0.999995500003375\n      ]\n    ],\n    "model_dim": 8,\n    "positions": [\n      0,\n      1,\n      2,\n      3\n    ]\n  },\n  "spectrum": {\n    "bases": {\n      "10000": {\n        "full_turn_counts": {\n          "131072": 32,\n          "4096": 23\n        },\n        "inverse_frequencies": [\n          1.0,\n          0.7498942093324558,\n          0.5623413251903491,\n          0.4216965034285822,\n          0.31622776601683794,\n          0.23713737056616552,\n          0.1778279410038923,\n          0.1333521432163324,\n          0.1,\n          0.07498942093324558,\n          0.056234132519034905,\n          0.042169650342858224,\n          0.03162277660168379,\n          0.023713737056616554,\n          0.01778279410038923,\n          0.01333521432163324,\n          0.01,\n          0.007498942093324558,\n          0.005623413251903491,\n          0.004216965034285822,\n          0.0031622776601683794,\n          0.0023713737056616554,\n          0.0017782794100389228,\n          0.001333521432163324,\n          0.001,\n          0.0007498942093324559,\n          0.0005623413251903491,\n          0.00042169650342858224,\n          0.00031622776601683794,\n          0.0002371373705661655,\n          0.00017782794100389227,\n          0.0001333521432163324\n        ],\n        "maximum_wavelength_tokens": 47117.2427801674,\n        "minimum_wavelength_tokens": 6.283185307179586,\n        "wavelengths": [\n          6.283185307179586,\n          8.378762269377678,\n          11.173259061216543,\n          14.899780425245321,\n          19.869176531592203,\n          26.49597274431474,\n          35.33294752055899,\n          47.117242780167395,\n          62.83185307179586,\n          83.78762269377677,\n          111.73259061216544,\n          148.99780425245322,\n          198.69176531592203,\n          264.95972744314736,\n          353.32947520558986,\n          471.1724278016739,\n          628.3185307179587,\n          837.8762269377677,\n          1117.3259061216543,\n          1489.9780425245322,\n          1986.9176531592202,\n          2649.597274431474,\n          3533.294752055899,\n          4711.724278016739,\n          6283.185307179586,\n          8378.762269377676,\n          11173.259061216542,\n          14899.78042524532,\n          19869.176531592202,\n          26495.972744314742,\n          35332.94752055899,\n          47117.2427801674\n        ]\n      },\n      "150000": {\n        "full_turn_counts": {\n          "131072": 27,\n          "4096": 18\n        },\n        "inverse_frequencies": [\n          1.0,\n          0.6890443058881632,\n          0.4747820554769007,\n          0.3271458718642365,\n          0.22541800020287084,\n          0.15532298948448497,\n          0.10702442147781144,\n          0.07374456821026082,\n          0.050813274815461475,\n          0.03501259767512414,\n          0.02412523106239743,\n          0.016623353091781194,\n          0.011454226792660225,\n          0.007892469749834168,\n          0.00543826134051781,\n          0.0037472030106155264,\n          0.0025819888974716113,\n          0.0017791047476692703,\n          0.0012258819959601083,\n          0.0008446870090071289,\n          0.0005820267738140659,\n          0.00040104223437104005,\n          0.00027633586801403137,\n          0.00019040765636773135,\n          0.00013119931141769537,\n          9.040213846881087e-05,\n          6.229107875204741e-05,\n          4.292131312172942e-05,\n          2.9574686407770566e-05,\n          2.0378269267702367e-05,\n          1.4041530402766067e-05,\n          9.675236569981486e-06\n        ],\n        "maximum_wavelength_tokens": 649408.9588128396,\n        "minimum_wavelength_tokens": 6.283185307179586,\n        "wavelengths": [\n          6.283185307179586,\n          9.118695638999144,\n          13.233830627546281,\n          19.206066307286523,\n          27.873485265262175,\n          40.452384595695705,\n          58.70795861748462,\n          85.20200822472704,\n          123.65243787176136,\n          179.4549883296344,\n          260.4404198628719,\n          377.9734011837886,\n          548.5473110420513,\n          796.0987506238595,\n          1155.3665618028806,\n          1676.7667215733509,\n          2433.467205584167,\n          3531.6556348977883,\n          5125.4405627016395,\n          7438.477495427609,\n          10795.354423311823,\n          15667.141185350692,\n          22737.494601535214,\n          32998.59589177952,\n          47890.3832595279,\n          69502.61811945203,\n          100868.14088081702,\n          146388.46880941882,\n          212451.46002727243,\n          308327.72031027393,\n          447471.54526274785,\n          649408.9588128396\n        ]\n      },\n      "500000": {\n        "full_turn_counts": {\n          "131072": 25,\n          "4096": 16\n        },\n        "inverse_frequencies": [\n          1.0,\n          0.6636012376960885,\n          0.44036660267178046,\n          0.2922278225730151,\n          0.19392274474868576,\n          0.12868737343265052,\n          0.08539710028576561,\n          0.05666962144529105,\n          0.03760603093086393,\n          0.024955408670558694,\n          0.016560440080994446,\n          0.010989528534539826,\n          0.007292664737217109,\n          0.004839421345719893,\n          0.003211445994752591,\n          0.0021311195369119653,\n          0.001414213562373095,\n          0.0009384738703573802,\n          0.000622772421914596,\n          0.0004132725499855165,\n          0.0002742481756762073,\n          0.00018199142881462546,\n          0.00012076973741146504,\n          8.01429472224798e-05,\n          5.318295896944988e-05,\n          3.529227739646723e-05,\n          2.341999896140934e-05,\n          1.5541540297632344e-05,\n          1.031338537721246e-05,\n          6.8439753011549275e-06,\n          4.5416704806078695e-06,\n          3.013858152139171e-06\n        ],\n        "maximum_wavelength_tokens": 2084764.7732591254,\n        "minimum_wavelength_tokens": 6.283185307179586,\n        "wavelengths": [\n          6.283185307179586,\n          9.468314629722128,\n          14.268078616903308,\n          21.500982527458316,\n          32.40045573469106,\n          48.8251888245115,\n          73.57609668424418,\n          110.87395939719457,\n          167.07919319459117,\n          251.77649423117703,\n          379.40931982782695,\n          571.7429357803371,\n          861.5760539647758,\n          1298.3340069648193,\n          1956.497265545218,\n          2948.3026167007256,\n          4442.882938158366,\n          6695.109481084312,\n          10089.055144515105,\n          15203.490547339246,\n          22910.58196353461,\n          34524.62211052572,\n          52026.15689866611,\n          78399.72854675821,\n          118142.83050307268,\n          178032.86641424094,\n          268282.9028956321,\n          404283.3069857812,\n          609226.2702664398,\n          918060.7805699259,\n          1383452.483839961,\n          2084764.7732591254\n        ]\n      }\n    },\n    "contexts": [\n      4096,\n      131072\n    ],\n    "head_dim": 64\n  },\n  "worked_rope": {\n    "base": 10000.0,\n    "head_dim": 4,\n    "k": [\n      0.0,\n      1.0,\n      1.0,\n      -0.5\n    ],\n    "logits": [\n      1.1442484284723262,\n      1.1442484284723262,\n      1.1442484284723264\n    ],\n    "placements": [\n      [\n        3,\n        1\n      ],\n      [\n        7,\n        5\n      ],\n      [\n        2,\n        0\n      ]\n    ],\n    "q": [\n      1.0,\n      0.0,\n      0.5,\n      0.5\n    ],\n    "rotated_k_at_1": [\n      -0.8414709848078965,\n      0.5403023058681398,\n      1.0049499170837486,\n      -0.489975166874166\n    ],\n    "rotated_q_at_3": [\n      -0.9899924966004454,\n      0.1411200080598672,\n      0.48477726677324595,\n      0.5147727669757416\n    ]\n  }\n}\n')
CONFIG_BYTES = b'{"num_hidden_layers": 36, "num_experts": 128, "experts_per_token": 4, "vocab_size": 201088, "hidden_size": 2880, "intermediate_size": 2880, "swiglu_limit": 7.0, "head_dim": 64, "num_attention_heads": 64, "num_key_value_heads": 8, "sliding_window": 128, "initial_context_length": 4096, "rope_theta": 150000, "rope_scaling_factor": 32.0, "rope_ntk_alpha": 1, "rope_ntk_beta": 32}'
assert hashlib.sha256(CONFIG_BYTES).hexdigest() == '063aaa98e012cac330d58d18471700f0b0eb90bb15c58fb227809d45090f3e65'
print("Source:", 'https://huggingface.co/openai/gpt-oss-120b/resolve/b5c939de8f754692c1647ca79fbf85e8c1e70f8a/original/config.json')
config = json.loads(CONFIG_BYTES)
sc = reference['scaling']
assert (config['head_dim'], config['rope_theta'], config['initial_context_length'], config['rope_scaling_factor']) == (sc['head_dim'], sc['base'], sc['original_context'], sc['factor'])
raw = rope_inverse_frequencies(config['head_dim'], config['rope_theta'])
yarn = yarn_scaled_frequencies(config['head_dim'], config['rope_theta'], config['rope_scaling_factor'], config['initial_context_length'])
np.testing.assert_allclose(yarn.inverse_frequencies, sc['yarn']['inverse_frequencies'], rtol=1e-12, atol=1e-15)
np.testing.assert_allclose(ntk_scaled_frequencies(sc['head_dim'],sc['base'],sc['factor']),sc['ntk'],rtol=1e-12,atol=1e-15)
scaling_plot(reference, DEMO_LANGUAGE)
plt.show()
print('YaRN bounds:', yarn.low, yarn.high)
print('Q/K amplitude and full-head score factor:', yarn.attention_factor, yarn.score_factor_if_applied_to_q_and_k)

Bounds near 8.0928 and 17.3980 yield 9 unchanged, 9 transitional, and 14 interpolated pairs. Applying an amplitude of 1.346574 to both full rotary vectors scales their dot product by about 1.813260. These results recover the positional geometry; the experiment does not measure long-text quality or establish the training history.

## Demo 2. Identify which axis changed

Sectioned M-RoPE assigns its coordinate pairs to particular axes. Change only the image width coordinate and inspect which output components move. Then give a text token equal IDs on all three axes and compare it with ordinary 1D RoPE. Matching frequencies and explicit tolerances let these comparisons test specific algebraic claims.

In [ ]:
mref = reference['mrope']
x = np.array(mref['image_input'])
coords = np.array(mref['image_coordinates'], dtype=np.int64)
inv = rope_inverse_frequencies(mref['head_dim'], mref['base'])
rotated = apply_mrope_adjacent(x, coords, mref['axis_sections'], inv)
np.testing.assert_allclose(rotated,mref['image_output'],rtol=1e-12,atol=1e-12)
np.testing.assert_allclose(np.linalg.norm(rotated,axis=-1),np.linalg.norm(x,axis=-1),rtol=1e-12)
changed_coords=coords.copy();changed_coords[:,2]+=1
width_changed=apply_mrope_adjacent(x,changed_coords,mref['axis_sections'],inv)
np.testing.assert_array_equal(width_changed[:,:6],rotated[:,:6])
assert not np.allclose(width_changed[:,6:],rotated[:,6:])
text_pos=np.array([4,5,6,7],dtype=np.int64)
text_axes=np.repeat(text_pos[:,None],3,axis=1)
np.testing.assert_allclose(apply_mrope_adjacent(x,text_axes,mref['axis_sections'],inv),apply_rope_adjacent(x,text_pos,inv),rtol=1e-12,atol=1e-12)
print('Image rotations:',rotated)
print('Only width pairs change; equal text IDs recover 1D RoPE: PASS')

With sections `(2,1,1)`, width owns the final pair, coordinates 6 and 7. Other components stay fixed, and rotations preserve vector norms. Equal text IDs explain the agreement with 1D RoPE. As a follow-up, change height instead of width and predict the affected pair before running the cell.

## Scaling to the full project

For a full project, pin the model and evaluation corpus. Verify pair layout and cache positions, then measure quality across lengths, memory, and runtime separately. Positional-vector similarity is not a language-quality metric.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.